In [36]:
%%capture

from collections import defaultdict
from seqeval.metrics import f1_score
from transformers import DataCollatorForTokenClassification
from transformers import Trainer
import import_ipynb
from a_glance_at_dataset_and_tokenizer import panx_ch, tags, xlmr_tokenizer
from create_model import device, tag_text, XLMRobertaForTokenClassification
from performance_measures import align_predictions
from tokenizing_text import encode_panx_dataset, panx_de_encoded

In [27]:
def compute_metrics(eval_pred):
    y_pred, y_true = align_predictions(eval_pred.predictions, eval_pred.label_ids)

    return {"f1": f1_score(y_true, y_pred)}

In [28]:
xlmr_model_name = "xlm-roberta-base"
model_name = f"{xlmr_model_name}-finetuned-panx-de"

In [29]:
model = XLMRobertaForTokenClassification.from_pretrained(
    model_name
).to(device)
data_collator = DataCollatorForTokenClassification(xlmr_tokenizer)

In [30]:
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=xlmr_tokenizer,
)

In [31]:
def get_f1_score(trainer, dataset):
    return trainer.predict(dataset).metrics["test_f1"]

In [32]:
f1_scores = defaultdict(dict)
f1_scores["de"]["de"] = get_f1_score(trainer, panx_de_encoded["test"])
print(f"F1 score of [de] model on [de] dataset: {f1_scores['de']['de']:.3f}")

F1 score of [de] model on [de] dataset: 0.866


In [34]:
text_fr = "Jeff Dean est informaticien chez Google en Californie"
tag_text(text_fr, tags, trainer.model, xlmr_tokenizer)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
Tokens,<s>,▁Jeff,▁De,an,▁est,▁informatic,ien,▁chez,▁Google,▁en,▁Cali,for,nie,</s>
Tags,O,B-PER,I-PER,I-PER,O,O,O,O,B-ORG,O,B-LOC,I-LOC,I-LOC,O


In [ ]:
def evaluate_lang_performance(lang, trainer):
    panx_ds = encode_panx_dataset